In [1]:

import os
import sys
import pickle
import numpy as np
import pandas as pd
import seaborn as sns

with open(sys.argv[0]) as f:
    code = f.read()
import uuid
from math import ceil

from itertools import product
from pathlib import Path
from typing import Tuple

import torch
from prettytable import PrettyTable
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
from torchvision.transforms.v2.functional import hflip
from timed_decorator.simple_timed import timed
from tqdm import tqdm

import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

torch.backends.cudnn.benchmark = True


In [2]:

LCL_PATH  = str(Path().cwd())
ROOT_PATH = str(Path(LCL_PATH).parent.parent)
DEEPL_PATH = str(Path(ROOT_PATH)/"deep_learning")
DS_PATH = "{}/data_visualisation/data".format(ROOT_PATH)

print("""
root path:\t{}
local path:\t{}
deep learning path:\t{}
dataset path:\t{}""".format(ROOT_PATH, LCL_PATH, DEEPL_PATH, DS_PATH))


root path:	/home/gheorghe/Desktop/Proiecte/master/CARN
local path:	/home/gheorghe/Desktop/Proiecte/master/CARN/laborator_3/test
deep learning path:	/home/gheorghe/Desktop/Proiecte/master/CARN/deep_learning
dataset path:	/home/gheorghe/Desktop/Proiecte/master/CARN/data_visualisation/data


In [3]:

# adding local_folder to the system path
sys.path.append(ROOT_PATH)
sys.path.append(LCL_PATH)
sys.path.append(DEEPL_PATH)

from sys_function import * # este in root

In [4]:

sys_remove_modules("imshow")
sys_remove_modules("dataset.row_dataset")
sys_remove_modules("dataset.append_help_dataset")
sys_remove_modules("dataset.add_virtual_dataset")
sys_remove_modules("dataloader.gpu_row_dataloader")
sys_remove_modules("trainer.trainer")
sys_remove_modules("trainer.whiten_trainer")
sys_remove_modules("trainer.supervised_callback_metrics_trainer")
sys_remove_modules("trainer.supervised_callback_metrics_trainer_two_opt")
sys_remove_modules("my_transformers.one_hot")
sys_remove_modules("my_transformers.label_smoothing")
sys_remove_modules("models.supervised.vgg13")
sys_remove_modules("models.supervised.se_st_resnext")
sys_remove_modules("models.supervised.apply_whiten2d")
sys_remove_modules("optimizer.muon")
sys_remove_modules("utils.utils")
sys_remove_modules("finetune.finetune_optimizer")
sys_remove_modules("finetune.finetune_tta")
sys_remove_modules("checks.tensor_check")
sys_remove_modules("metrics.accuracy")
sys_remove_modules("callback.save_best_acc_val")
sys_remove_modules("callback.save_history")
sys_remove_modules("callback.time_acc")
sys_remove_modules("data_visualisation", "time_mon_load_data")

from imshow import *
from dataset.row_dataset import *
from dataset.append_help_dataset import *
from dataset.add_virtual_dataset import *
from dataloader.gpu_row_dataloader import *
from trainer.trainer import *
from trainer.whiten_trainer import *
from trainer.supervised_callback_metrics_trainer import *
from trainer.supervised_callback_metrics_trainer_two_opt import *
from my_transformers.one_hot import *
from my_transformers.label_smoothing import *
from models.supervised.vgg13 import *
from models.supervised.se_st_resnext import *
from models.supervised.apply_whiten2d import *
from optimizer.muon import *
from utils.utils import *
from finetune.finetune_optimizer import *
from finetune.finetune_tta import *
from checks.tensor_check import *
from metrics.accuracy import *
from callback.save_best_acc_val import *
from callback.save_history import *
from callback.time_acc import *
from data_visualisation.time_mon_load_data import *

In [5]:

BATCH_SIZE = 2000
IMAGE_SIZE = 32

## Data aquisition

In [6]:

if os.path.exists("/kaggle/input") and os.path.exists("/kaggle/working"):
    print("Running on Kaggle.")
    file_SVHN_test = "/kaggle/input/fii-atnn-2025-competition-2/SVHN_test.pkl"
    file_SVHN_train = "/kaggle/input/fii-atnn-2025-competition-2/SVHN_train.pkl"
else:
    print("Not on Kaggle.")
    file_SVHN_test = "{}/fii-atnn-2025-competition-2/SVHN_test.pkl".format(DS_PATH)
    file_SVHN_train = "{}/fii-atnn-2025-competition-2/SVHN_train.pkl".format(DS_PATH)

with open(file_SVHN_train, "rb") as fd:
    train_in, train_out = list(zip(*pickle.load(fd)))
    train_in, train_out = np.array(train_in, dtype=np.uint8), np.array(train_out, dtype=np.int64)

with open(file_SVHN_test, "rb") as fd:
    test_in,  test_out  = list(zip(*pickle.load(fd)))
    test_in,  test_out  = np.array(test_in, dtype=np.uint8),  np.array(test_out, dtype=np.int64)
    
mean, std = train_in.mean(axis=(0, 1, 2)), train_in.std(axis=(0, 1, 2))
print("mean {}, std {}".format(mean, std))

svhn_train_ds = RowDataset(dict(inputs=train_in, targets=train_out), name="SVHN train")
svhn_test_ds  = RowDataset(dict(inputs=test_in, targets=test_out), name="SVHN test")
print(svhn_train_ds)
print(svhn_test_ds)
NUM_CLASSES = train_out.max()+1

Not on Kaggle.
mean [129.30416561 124.0699627  112.43405006], std [68.1702429  65.39180804 70.41837019]
RowDataset:
   name: SVHN train,
   size: 50000,
   num_classes : 100,
   shape: (50000, 32, 32, 3),
   in min: 0,
   in max: 255,
   out min: 0,
   out max: 99,
RowDataset:
   name: SVHN test,
   size: 10000,
   num_classes : 100,
   shape: (10000, 32, 32, 3),
   in min: 0,
   in max: 255,
   out min: 0,
   out max: 99,


In [8]:
np.unique(train_out, return_counts=True)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
        51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
        68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
        85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]),
 array([500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500,
        500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500,
        500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500,
        500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500,
        500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500,
        500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500,
        500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500,
        500, 500, 500, 500, 500, 500, 5

In [7]:
  
def get_init_transform(train, image_size, mean, std):
    transform = [ # image->tensor->resize->make square-> 
            # if use 'ToImage' tensor should be numpy array!!!
            v2.ToImage(), # data are transorm to torch tensor in Dataset manager, tensor should be numpy array!!!
            v2.Resize(
                size=int(image_size),),
            v2.CenterCrop(image_size),
        ]
    if (train == False):
        transform.extend([
                v2.ToDtype(torch.float16, scale=False), # scale True normalized
                v2.Normalize(mean=mean, std=std, inplace=True),
            ])
                # We use the inplace flag because we can safely change the tensors inplace when normalize is used.
                # For is_train=False, we can safely change the tensors inplace because we do it only once, when caching.
                # For is_train=True, we can safely change the tensors inplace because we clone the cached tensors first.
    return v2.Compose(transform)

### Load Data

In [8]:

# prepare meta data for training
svhn_train_meta_ds = dict(data_reader=svhn_train_ds, size=50000, num_classes=NUM_CLASSES)
svhn_test_meta_ds  = dict(data_reader=svhn_test_ds,  size=10000, num_classes=NUM_CLASSES)

train_ds = AppendHelpDatasets(svhn_train_meta_ds, help_metads=None, 
                              transform=get_init_transform(False, IMAGE_SIZE, mean, std), 
                              help_from=0.1)
test_ds  = AppendHelpDatasets(svhn_test_meta_ds, help_metads=None, 
                              transform=get_init_transform(False, IMAGE_SIZE, mean, std))

print("size: train_ds {}, test_ds {}".format(len(train_ds), len(test_ds)))

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
test_dl  = DataLoader(test_ds , batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)

size: train_ds 50000, test_ds 10000


### Load to GPU SVHN 100

In [32]:

path = "{}/svhn_air_bench".format(DS_PATH)
#train_in = np.concatenate(
MEAN_DS, STD_DS = train_in.mean(axis=(0, 1, 2)), train_in.std(axis=(0, 1, 2))
GpuRowDataLoader(path, train_in, train_out, True,  MEAN_DS, STD_DS, batch_size=500, aug=None)
GpuRowDataLoader(path, test_in,  test_out,  False, MEAN_DS, STD_DS, batch_size=500, aug=None)

### Load to GPU Cifar 10

In [10]:

path = "{}/cifar_air_bench".format(DS_PATH)
dset = torchvision.datasets.CIFAR10(path, download=True, train=True)
inputs  = np.array(dset.data)
targets = np.array(dset.targets)
MEAN_DS, STD_DS = inputs.mean(axis=(0, 1, 2)), inputs.std(axis=(0, 1, 2))
print(type(inputs), inputs.dtype, inputs.shape, type(targets), len(targets), type(targets[0]))
GpuRowDataLoader(path, inputs, targets, True, MEAN_DS, STD_DS, batch_size=500, aug=None)


Files already downloaded and verified
<class 'numpy.ndarray'> uint8 (50000, 32, 32, 3) <class 'numpy.ndarray'> 50000 <class 'numpy.int64'>


### Load to GPU Mix SVHN and Cifar 10

In [34]:

path = "{}/cifar_air_bench".format(DS_PATH)
dset = torchvision.datasets.CIFAR10(path, download=True, train=False)
cifar_in  = np.array(dset.data)
cifar_out = np.array(dset.targets)

path = "{}/mix_svhn_cifar10_air_bench".format(DS_PATH)
train_in  = np.concatenate((train_in, cifar_in), axis=0)
train_out = np.concatenate((train_out, cifar_out), axis=0)
GpuRowDataLoader(path, train_in, train_out, True,  MEAN_DS, STD_DS, batch_size=500, aug=None)
GpuRowDataLoader(path, test_in,  test_out,  False, MEAN_DS, STD_DS, batch_size=500, aug=None)

Files already downloaded and verified


## Build model

In [13]:

#############################################
#            Network Definition             #
#############################################

# note the use of low BatchNorm stats momentum
class BatchNorm(nn.BatchNorm2d):
    def __init__(self, num_features, momentum=0.6, eps=1e-12):
        super().__init__(num_features, eps=eps, momentum=1-momentum)
        self.weight.requires_grad = False
        # Note that PyTorch already initializes the weights to one and bias to zero

class Conv(nn.Conv2d):
    def __init__(self, in_channels, out_channels):
        super().__init__(in_channels, out_channels, kernel_size=3, padding="same", bias=False)

    def reset_parameters(self):
        super().reset_parameters()
        w = self.weight.data
        torch.nn.init.dirac_(w[:w.size(1)])

class ConvGroup(nn.Module):
    def __init__(self, channels_in, channels_out):
        super().__init__()
        self.conv1 = Conv(channels_in,  channels_out)
        self.pool = nn.MaxPool2d(2)
        self.norm1 = BatchNorm(channels_out)
        self.conv2 = Conv(channels_out, channels_out)
        self.norm2 = BatchNorm(channels_out)
        self.activ = nn.GELU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool(x)
        x = self.norm1(x)
        x = self.activ(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.activ(x)
        return x

class CifarNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        widths = dict(block1=64, block2=256, block3=256)
        whiten_kernel_size = 2
        whiten_width = 2 * 3 * whiten_kernel_size**2
        self.whiten = nn.Conv2d(3, whiten_width, whiten_kernel_size, padding=0, bias=True)
        self.whiten.weight.requires_grad = False
        self.layers = nn.Sequential(
            nn.GELU(),
            ConvGroup(whiten_width,     widths["block1"]),
            ConvGroup(widths["block1"], widths["block2"]),
            ConvGroup(widths["block2"], widths["block3"]),
            nn.MaxPool2d(3),
        )
        self.head = nn.Linear(widths["block3"], int(num_classes), bias=False)
        for mod in self.modules():
            if isinstance(mod, BatchNorm):
                mod.float()
            else:
                mod.half()

    def reset_parameters(self):
        for m in self.modules():
            if type(m) in (nn.Conv2d, Conv, BatchNorm, nn.Linear):
                m.reset_parameters()
        w = self.head.weight.data
        w *= 1 / w.std()

    def init_whiten(self, train_images, eps=5e-4):
        c, (h, w) = train_images.shape[1], self.whiten.weight.shape[2:]
        patches = train_images.unfold(2,h,1).unfold(3,w,1).transpose(1,3).reshape(-1,c,h,w).float()
        patches_flat = patches.view(len(patches), -1)
        est_patch_covariance = (patches_flat.T @ patches_flat) / len(patches_flat)
        eigenvalues, eigenvectors = torch.linalg.eigh(est_patch_covariance, UPLO="U")
        eigenvectors_scaled = eigenvectors.T.reshape(-1,c,h,w) / torch.sqrt(eigenvalues.view(-1,1,1,1) + eps)
        self.whiten.weight.data[:] = torch.cat((eigenvectors_scaled, -eigenvectors_scaled))

    def forward(self, x, whiten_bias_grad=True):
        b = self.whiten.bias
        x = F.conv2d(x, self.whiten.weight, b if whiten_bias_grad else b.detach())
        x = self.layers(x)
        x = x.view(len(x), -1)
        return self.head(x) / x.size(-1)


## Version 0

In [26]:
"""
airbench94_muon.py
Runs in 2.59 seconds on a 400W NVIDIA A100 using torch==2.4.1
Attains 94.01 mean accuracy (n=200 trials)
Descends from https://github.com/tysam-code/hlb-CIFAR10/blob/main/main.py
"""

############################################
#                 Logging                  #
############################################

def print_columns(columns_list, is_head=False, is_final_entry=False):
    print_string = ""
    for col in columns_list:
        print_string += "|  %s  " % col
    print_string += "|"
    if is_head:
        print("-"*len(print_string))
    print(print_string)
    if is_head or is_final_entry:
        print("-"*len(print_string))

logging_columns_list = ["run   ", "epoch", "train_acc", "val_acc", "tta_val_acc", "time_seconds"]
def print_training_details(variables, is_final_entry):
    formatted = []
    for col in logging_columns_list:
        var = variables.get(col.strip(), None)
        if type(var) in (int, str):
            res = str(var)
        elif type(var) is float:
            res = "{:0.4f}".format(var)
        else:
            assert var is None
            res = ""
        formatted.append(res.rjust(len(col)))
    print_columns(formatted, is_final_entry=is_final_entry)

############################################
#               Evaluation                 #
############################################

def infer(model, loader, tta_level=0):

    # Test-time augmentation strategy (for tta_level=2):
    # 1. Flip/mirror the image left-to-right (50% of the time).
    # 2. Translate the image by one pixel either up-and-left or down-and-right (50% of the time,
    #    i.e. both happen 25% of the time).
    #
    # This creates 6 views per image (left/right times the two translations and no-translation),
    # which we evaluate and then weight according to the given probabilities.

    def infer_basic(inputs, net):
        return net(inputs).clone()

    def infer_mirror(inputs, net):
        return 0.5 * net(inputs) + 0.5 * net(inputs.flip(-1))

    def infer_mirror_translate(inputs, net):
        logits = infer_mirror(inputs, net)
        pad = 1
        padded_inputs = F.pad(inputs, (pad,)*4, "reflect")
        inputs_translate_list = [
            padded_inputs[:, :, 0:32, 0:32],
            padded_inputs[:, :, 2:34, 2:34],
        ]
        logits_translate_list = [infer_mirror(inputs_translate, net)
                                 for inputs_translate in inputs_translate_list]
        logits_translate = torch.stack(logits_translate_list).mean(0)
        return 0.5 * logits + 0.5 * logits_translate

    model.eval()
    test_images = loader.normalize(loader.images)
    infer_fn = [infer_basic, infer_mirror, infer_mirror_translate][tta_level]
    with torch.no_grad():
        return torch.cat([infer_fn(inputs, model) for inputs in test_images.split(2000)])

def evaluate(model, loader, tta_level=0):
    logits = infer(model, loader, tta_level)
    return (logits.argmax(1) == loader.labels).float().mean().item()

############################################
#                Training                  #
############################################

def main(run, model):

    batch_size = 2000
    bias_lr = 0.053
    head_lr = 0.67
    wd = 2e-6 * batch_size
    
    path = "{}/svhn_air_bench".format(DS_PATH)
    #test_dl  = MyDataLoader(path, test_in,  test_out,  train=False, batch_size=batch_size, aug=None)
    #train_dl = MyDataLoader(path, train_in, train_out, train=True, batch_size=batch_size, aug=dict(flip=True, translate=2))
    test_dl  = GpuRowDataLoader(path, test_in,  test_out,  False, MEAN_DS, STD_DS, batch_size=2000, aug=None)
    train_dl = GpuRowDataLoader(path, train_in, train_out, True,  MEAN_DS, STD_DS, batch_size=2000, aug=dict(flip=True, translate=2))

    if (run == "warmup"):
        # The only purpose of the first run is to warmup the compiled model, so we can use dummy data
        train_dl.labels = torch.randint(0, train_dl.labels.max()+1, size=(len(train_dl.labels),), device=train_dl.labels.device)
    
    total_train_steps = ceil(8 * len(train_dl))
    whiten_bias_train_steps = ceil(3 * len(train_dl))

    # Create optimizers and learning rate schedulers
    filter_params = [p for p in model.parameters() if len(p.shape) == 4 and p.requires_grad]
    norm_biases   = [p for n, p in model.named_parameters() if "norm" in n and p.requires_grad]
    param_configs = [dict(params=[model.whiten.bias], lr=bias_lr, weight_decay=wd/bias_lr),
                     dict(params=norm_biases,         lr=bias_lr, weight_decay=wd/bias_lr),
                     dict(params=[model.head.weight], lr=head_lr, weight_decay=wd/head_lr)]
    optimizer1 = torch.optim.SGD(param_configs, momentum=0.85, nesterov=True)
    optimizer2 = Muon(filter_params, lr=0.24, momentum=0.6, nesterov=True, ns_steps=6)
    optimizers = [optimizer1, optimizer2]
    for opt in optimizers:
        for group in opt.param_groups:
            group["initial_lr"] = group["lr"]

    # For accurately timing GPU code
    starter = torch.cuda.Event(enable_timing=True)
    ender   = torch.cuda.Event(enable_timing=True)
    time_seconds = 0.0
    def start_timer():
        starter.record()

    def stop_timer():
        ender.record()
        torch.cuda.synchronize()
        nonlocal time_seconds
        time_seconds += 1e-3 * starter.elapsed_time(ender)

    model.reset_parameters()
    step = 0

    # Initialize the whitening layer using training images
    start_timer()
    train_images = train_dl.normalize(train_dl.images[:5000])
    model.init_whiten(train_images)
    stop_timer()

    for epoch in range(ceil(total_train_steps / len(train_dl))):

        ####################
        #     Training     #
        ####################

        start_timer()
        model.train()
        for inputs, labels in train_dl:
            outputs = model(inputs, whiten_bias_grad=(step < whiten_bias_train_steps))
            
            F.cross_entropy(outputs, labels, label_smoothing=0.2, reduction="sum").backward()
            for group in optimizer1.param_groups[:1]:
                group["lr"] = group["initial_lr"] * (1 - step / whiten_bias_train_steps)
            for group in optimizer1.param_groups[1:]+optimizer2.param_groups:
                group["lr"] = group["initial_lr"] * (1 - step / total_train_steps)
            for opt in optimizers:
                opt.step()
            model.zero_grad(set_to_none=True)
            step += 1
            if step >= total_train_steps:
                break
        stop_timer()

        ####################
        #    Evaluation    #
        ####################

        # Save the accuracy and loss from the last training batch of the epoch
        train_acc = (outputs.detach().argmax(1) == labels).float().mean().item()
        val_acc = evaluate(model, test_dl, tta_level=0)
        print_training_details(locals(), is_final_entry=False)
        run = None # Only print the run number once

    ####################
    #  TTA Evaluation  #
    ####################

    start_timer()
    tta_val_acc = evaluate(model, test_dl, tta_level=2)
    stop_timer()
    epoch = "eval"
    print_training_details(locals(), is_final_entry=True)

    return tta_val_acc

if __name__ == "__main__":

    # We re-use the compiled model between runs to save the non-data-dependent compilation time
    model = CifarNet(100).cuda().to(memory_format=torch.channels_last)
    
    #model = VGG13(24, train_ds.num_classes)
    #model = ApplyWhiten2d(model, 3).cuda().to(memory_format=torch.channels_last)
    
    model.compile(mode="max-autotune")
    #model.compile()

    print_columns(logging_columns_list, is_head=True)
    main("warmup", model)
    accs = torch.tensor([main(run, model) for run in range(6)])
    print("Mean: %.4f    Std: %.4f" % (accs.mean(), accs.std()))

    log_dir = os.path.join("logs", str(uuid.uuid4()))
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, "log.pt")
    torch.save(dict(code=code, accs=accs), log_path)
    print(os.path.abspath(log_path))

---------------------------------------------------------------------------------
|  run     |  epoch  |  train_acc  |  val_acc  |  tta_val_acc  |  time_seconds  |
---------------------------------------------------------------------------------
|  warmup  |      0  |     0.0145  |   0.0028  |               |        0.4868  |
|          |      1  |     0.0060  |   0.0071  |               |        0.9492  |
|          |      2  |     0.0075  |   0.0059  |               |        1.4097  |
|          |      3  |     0.0115  |   0.0095  |               |        1.8351  |
|          |      4  |     0.0120  |   0.0065  |               |        2.2602  |
|          |      5  |     0.0115  |   0.0092  |               |        2.6857  |
|          |      6  |     0.0130  |   0.0096  |               |        3.1143  |
|          |      7  |     0.0135  |   0.0079  |               |        3.5401  |
|          |   eval  |     0.0135  |   0.0079  |       0.0071  |        3.7164  |
----------------

## Version 1

In [42]:
"""
airbench94_muon.py
Runs in 2.59 seconds on a 400W NVIDIA A100 using torch==2.4.1
Attains 94.01 mean accuracy (n=200 trials)
Descends from https://github.com/tysam-code/hlb-CIFAR10/blob/main/main.py
"""

############################################
#                 Logging                  #
############################################

def print_columns(columns_list, is_head=False, is_final_entry=False):
    print_string = ""
    for col in columns_list:
        print_string += "|  %s  " % col
    print_string += "|"
    if is_head:
        print("-"*len(print_string))
    print(print_string)
    if is_head or is_final_entry:
        print("-"*len(print_string))

logging_columns_list = ["run   ", "epoch", "train_acc", "val_acc", "tta_val_acc", "time_seconds"]
def print_training_details(variables, is_final_entry):
    formatted = []
    for col in logging_columns_list:
        var = variables.get(col.strip(), None)
        if type(var) in (int, str):
            res = str(var)
        elif type(var) is float:
            res = "{:0.4f}".format(var)
        else:
            assert var is None
            res = ""
        formatted.append(res.rjust(len(col)))
    print_columns(formatted, is_final_entry=is_final_entry)

############################################
#               Evaluation                 #
############################################

def infer(model, loader, tta_level=0):

    # Test-time augmentation strategy (for tta_level=2):
    # 1. Flip/mirror the image left-to-right (50% of the time).
    # 2. Translate the image by one pixel either up-and-left or down-and-right (50% of the time,
    #    i.e. both happen 25% of the time).
    #
    # This creates 6 views per image (left/right times the two translations and no-translation),
    # which we evaluate and then weight according to the given probabilities.

    def infer_basic(inputs, net):
        return net(inputs).clone()

    def infer_mirror(inputs, net):
        return 0.5 * net(inputs) + 0.5 * net(inputs.flip(-1))

    def infer_mirror_translate(inputs, net):
        logits = infer_mirror(inputs, net)
        pad = 1
        padded_inputs = F.pad(inputs, (pad,)*4, "reflect")
        inputs_translate_list = [
            padded_inputs[:, :, 0:32, 0:32],
            padded_inputs[:, :, 2:34, 2:34],
        ]
        logits_translate_list = [infer_mirror(inputs_translate, net)
                                 for inputs_translate in inputs_translate_list]
        logits_translate = torch.stack(logits_translate_list).mean(0)
        return 0.5 * logits + 0.5 * logits_translate

    model.eval()
    test_images = loader.normalize(loader.images)
    infer_fn = [infer_basic, infer_mirror, infer_mirror_translate][tta_level]
    with torch.no_grad():
        return torch.cat([infer_fn(inputs, model) for inputs in test_images.split(2000)])

def evaluate(model, loader, tta_level=0):
    logits = infer(model, loader, tta_level)
    return (logits.argmax(1) == loader.labels).float().mean().item()

############################################
#                Training                  #
############################################

def main(run, model, train_dl, test_dl):

    batch_size = 2000
    bias_lr = 0.003
    head_lr = 0.97
    wd = 2e-6 * batch_size

    train_dl.setWorkMode(run)
    
    total_train_steps = ceil(8 * len(train_dl))
    whiten_bias_train_steps = ceil(3 * len(train_dl))

    # Create optimizers and learning rate schedulers
    filter_params = [p for p in model.parameters() if len(p.shape) == 4 and p.requires_grad]
    norm_biases   = [p for n, p in model.named_parameters() if "norm" in n and p.requires_grad]
    param_configs = [dict(params=[model.whiten.bias], lr=bias_lr, weight_decay=wd/bias_lr),
                     dict(params=norm_biases,         lr=bias_lr, weight_decay=wd/bias_lr),
                     dict(params=[model.head.weight], lr=head_lr, weight_decay=wd/head_lr)]
    optimizer1 = torch.optim.SGD(param_configs, momentum=0.85, nesterov=True)
    optimizer2 = Muon(filter_params, lr=0.24, momentum=0.6, nesterov=True, ns_steps=6)
    optimizers = [optimizer1, optimizer2]
    for opt in optimizers:
        for group in opt.param_groups:
            group["initial_lr"] = group["lr"]

    # For accurately timing GPU code
    starter = torch.cuda.Event(enable_timing=True)
    ender   = torch.cuda.Event(enable_timing=True)
    time_seconds = 0.0
    def start_timer():
        starter.record()

    def stop_timer():
        ender.record()
        torch.cuda.synchronize()
        nonlocal time_seconds
        time_seconds += 1e-3 * starter.elapsed_time(ender)

    model.reset_parameters()
    step = 0

    # Initialize the whitening layer using training images
    start_timer()
    train_images = train_dl.normalize(train_dl.images[:5000])
    model.init_whiten(train_images)
    stop_timer()

    for epoch in range(ceil(total_train_steps / len(train_dl))):

        ####################
        #     Training     #
        ####################

        start_timer()
        model.train()
        for inputs, labels in train_dl:
            outputs = model(inputs, whiten_bias_grad=(step < whiten_bias_train_steps))
            
            F.cross_entropy(outputs, labels, label_smoothing=0.2, reduction="sum").backward()
            for group in optimizer1.param_groups[:1]:
                group["lr"] = group["initial_lr"] * (1 - step / whiten_bias_train_steps)
            for group in optimizer1.param_groups[1:]+optimizer2.param_groups:
                group["lr"] = group["initial_lr"] * (1 - step / total_train_steps)
            for opt in optimizers:
                opt.step()
            model.zero_grad(set_to_none=True)
            step += 1
            if step >= total_train_steps:
                break
        stop_timer()

        ####################
        #    Evaluation    #
        ####################

        # Save the accuracy and loss from the last training batch of the epoch
        train_acc = (outputs.detach().argmax(1) == labels).float().mean().item()
        val_acc = evaluate(model, test_dl, tta_level=0)
        print_training_details(locals(), is_final_entry=False)
        run = None # Only print the run number once

    ####################
    #  TTA Evaluation  #
    ####################

    start_timer()
    tta_val_acc = evaluate(model, test_dl, tta_level=2)
    stop_timer()
    epoch = "eval"
    print_training_details(locals(), is_final_entry=True)

    return tta_val_acc

if __name__ == "__main__":

    # We re-use the compiled model between runs to save the non-data-dependent compilation time
    #model = CifarNet(110).cuda().to(memory_format=torch.channels_last)
    
    model = VGG13(24, train_ds.num_classes)
    model = ApplyWhiten2d(model, 3).cuda().to(memory_format=torch.channels_last)
    
    model.compile(mode="max-autotune")
    #model.compile()
    
    path = "{}/svhn_air_bench".format(DS_PATH)
    #MEAN_DS, STD_DS = train_in.mean(axis=(0, 1, 2)), train_in.std(axis=(0, 1, 2))
    test_dl  = GpuRowDataLoader(path, test_in,  test_out,  False, MEAN_DS, STD_DS, batch_size=2000, aug=None)
    train_dl = GpuRowDataLoader(path, train_in, train_out, True,  MEAN_DS, STD_DS, batch_size=2000, aug=dict(flip=True, translate=2))

    print_columns(logging_columns_list, is_head=True)
    #main("warmup", model, train_dl, test_dl)
    accs = torch.tensor([main(run, model, train_dl, test_dl) for run in range(6)])
    print("Mean: %.4f    Std: %.4f" % (accs.mean(), accs.std()))

    log_dir = os.path.join("logs", str(uuid.uuid4()))
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, "log.pt")
    torch.save(dict(code=code, accs=accs), log_path)
    print(os.path.abspath(log_path))

---------------------------------------------------------------------------------
|  run     |  epoch  |  train_acc  |  val_acc  |  tta_val_acc  |  time_seconds  |
---------------------------------------------------------------------------------
|       0  |      0  |     0.1630  |   0.1433  |               |        1.0787  |
|          |      1  |     0.2920  |   0.1821  |               |        2.1272  |
|          |      2  |     0.3995  |   0.3339  |               |        3.1755  |
|          |      3  |     0.4700  |   0.4558  |               |        4.1960  |
|          |      4  |     0.5495  |   0.4938  |               |        5.2254  |
|          |      5  |     0.6550  |   0.5772  |               |        6.2657  |
|          |      6  |     0.7210  |   0.6377  |               |        7.3467  |
|          |      7  |     0.7715  |   0.6785  |               |        8.4087  |
|          |   eval  |     0.7715  |   0.6785  |       0.6940  |        8.8486  |
----------------